# Plan D2/D3: fine-tune ladder on canonical recipe

Requires D1 done (`out/dapt-afro-xlmr/` exists on Drive). Free T4 runtime.

- `run_d_batch.sh` is idempotent: finished variants skip, interrupted ones resume.
  Re-run the batch cell after every disconnect. ~2-2.5h per variant, 4 variants.
- Variant order = priority order (d-base + d-dapt answer the DAPT question first).
- Stage 2 (llrd on best) and stage 3 (seeds) scripts get authored after stage-1
  metrics are read - see the commented cells.

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%pip install -q "transformers>=4.48" "accelerate>=1.0" "scikit-learn>=1.4" "pandas>=2" "pyarrow>=18" matplotlib

In [ ]:
# stage 1 ladder - safe to re-run any time
!cd /content/drive/MyDrive/Colab/hatespeech-finetune && bash run_d_batch.sh

In [ ]:
import json, glob
for p in sorted(glob.glob('/content/drive/MyDrive/Colab/hatespeech-finetune/out/eval-d*_metrics.json')):
    m = json.load(open(p))
    name = p.split('/')[-1].replace('_metrics.json', '')
    print(f"{name:32s} macro_f1 {m['macro_f1']:.4f}")

In [ ]:
# stage 2 (llrd on best of ls/focal) - uncomment once run_d2_batch.sh exists
# !cd /content/drive/MyDrive/Colab/hatespeech-finetune && bash run_d2_batch.sh

In [ ]:
# stage 3 (multi-seed confirm) - uncomment once run_d3_seeds.sh exists
# !cd /content/drive/MyDrive/Colab/hatespeech-finetune && bash run_d3_seeds.sh

In [ ]:
# push final winner (fill in the model dir once chosen; needs HF_TOKEN secret)
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
!pip install -q huggingface_hub
# !cd /content/drive/MyDrive/Colab/hatespeech-finetune && python 10_push_hf.py --model-dir out/model-<WINNER> --repo-id kenya-hatespeech-afroxlmr